# From a linear program to PyPSA: the same problem three times, then one you cannot check by hand

Every capacity-expansion model in an energy course is a linear program with a lot of bookkeeping. PyPSA
does the bookkeeping. Before trusting it with two thousand buses, it is worth seeing exactly what it
writes for one — so this notebook solves one hour of dispatch on paper, then in gurobipy where every
row is a line you typed, then in PyPSA where you describe the *system* and it writes the rows. All
three must agree.

Then the problem grows: twenty-four hours, solar that follows the sun, two thermal units, and a
battery that links every hour to the next. Then capacity itself becomes a decision. At each step the
model does nothing you could not have written yourself; it just stops being something you would want
to.

## Setup: where the package lives, and PyPSA

This notebook builds its models by hand and then checks them against `orteach`, the package in
`src/`. It also needs PyPSA, which does not live in the base environment on the authoring machine —
see the README in this folder for the environment and kernel. On Colab this cell installs it.

In [1]:
import os, subprocess, sys

REPO_URL = None      # the public GitHub URL, once this library is published; Colab clones from it

try:
    import google.colab                      # noqa: F401 - succeeds only on Colab
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if ON_COLAB:
    if REPO_URL is None:
        raise SystemExit("This library is not published yet: open the notebook from a clone of the repository.")
    if not os.path.isdir("/content/teaching-code"):
        subprocess.run(["git", "clone", "--quiet", REPO_URL, "/content/teaching-code"], check=True)
    os.chdir("/content/teaching-code/notebooks/12_energy_systems_pypsa")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "gurobipy>=11,<14", "pypsa", "highspy"], check=True)

sys.path.insert(0, os.path.abspath(os.path.join("..", "..", "src")))
try:
    import orteach                            # noqa: F401
except ImportError:
    raise SystemExit("orteach not found: run this notebook from its own folder inside the repository, "
                     "so that ../../src exists.")
try:
    import pypsa
except ImportError:
    raise SystemExit("PyPSA is not installed in this kernel. See README.md in this folder for the orteach-energy environment.")
print("package:", os.path.dirname(orteach.__file__), "  pypsa", pypsa.__version__)

package: C:\Users\jonesec\OneDrive - UT Arlington\Documents\Classes\Teaching Code\src\orteach   pypsa 1.3.0


## Licence setup, and a quiet solver

Three secrets named, none contained. Colab reads them from the key icon in the left sidebar; a
machine with a licence file needs nothing. The environment starts silent, so the licence number
never lands in an output cell. PyPSA and linopy log every step of building a model; that is turned
down to errors, and pandas 3's deprecation warnings inside PyPSA are silenced, so the outputs below
are the numbers and nothing else.

In [2]:
import logging, warnings
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
from orteach import tolerance

env = gp.Env(empty=True)
env.setParam("OutputFlag", 0)        # start silent: the licence banner, and its licence number, stay out of the outputs
try:
    from google.colab import userdata
    try:
        env.setParam("WLSACCESSID", userdata.get("GRB_WLSACCESSID"))
        env.setParam("WLSSECRET",   userdata.get("GRB_WLSSECRET"))
        env.setParam("LICENSEID",   int(userdata.get("GRB_LICENSEID")))
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        raise SystemExit("Add GRB_WLSACCESSID, GRB_WLSSECRET and GRB_LICENSEID as Colab Secrets "
                         "(key icon, left sidebar), grant this notebook access to them, then re-run this cell.")
    env.start()
    print("licence: Colab Secrets (WLS)")
except ImportError:
    env.start()
    print("licence: local gurobi.lic")

logging.getLogger("pypsa").setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)
# every PyPSA solve below goes through Gurobi with the package's tightened tolerances, silently
SOLVE = dict(solver_name="gurobi", env=env, solver_options=dict(tolerance.GUROBI_PARAMS))

licence: local gurobi.lic


## Part A — one hour, on paper

One hour. One place. **100 MW** is needed. Two things can supply it:

| technology | available this hour | cost to run |
|---|---|---|
| solar | 40 MW | $0.00 /MWh |
| gas | 500 MW | $22.14 /MWh |

**Solve it before you run anything below**: which do you run, how much of each, and what does the
hour cost? Then write it as a linear program — one variable per technology, a balance row, a
capacity row each — because the same five parts (sets, parameters, variables, objective,
constraints) will carry through every model in this notebook.

## Part B — the same LP in gurobipy

Two technologies with two numbers each, and a demand: these are knobs, written out, because the
story names every one of them.

In [3]:
AVAILABLE = {"solar": 40.0, "gas": 500.0}      # MW this hour
COST      = {"solar": 0.00, "gas": 22.14}      # $/MWh to run
DEMAND    = 100.0                               # MW

m = gp.Model("one hour", env=env)
tolerance.apply(m)
p = m.addVars(AVAILABLE.keys(), lb=0.0, name="p")                         # MW from each technology
m.setObjective(gp.quicksum(COST[g] * p[g] for g in AVAILABLE), GRB.MINIMIZE)
balance = m.addConstr(p.sum() == DEMAND, name="balance")
capacity = m.addConstrs((p[g] <= AVAILABLE[g] for g in AVAILABLE), name="capacity")
m.update()
print(m.NumVars, "variables,", m.NumConstrs, "constraints")

2 variables, 3 constraints


Predict before you run: the cost of the hour, and the shadow price on the balance row — the cost of
one more MW of demand.

In [4]:
m.optimize()
lp_cost, lp_price = m.ObjVal, balance.Pi
lp_p = {g: p[g].X for g in AVAILABLE}
print(f"objective ${lp_cost:,.2f}")
for g in AVAILABLE:
    print(f"  {g:6} {lp_p[g]:6.1f} MW")
print(f"shadow price on balance: ${lp_price:.2f}/MWh")

objective $1,328.40
  solar    40.0 MW
  gas      60.0 MW
shadow price on balance: $22.14/MWh


The price is gas's cost, not an average of what ran. Nothing was added to the model to get that;
say where it comes from.

## Part C — the same problem in PyPSA

Now describe the *system* rather than the LP. One component per cell, each with the LP symbol it
stands for. One hour, so one snapshot; one place, so one bus — and the bus is where the balance row
lives.

In [5]:
n = pypsa.Network()
n.set_snapshots([0])
n.add("Bus", "node")
print("buses:", list(n.buses.index), "  snapshots:", list(n.snapshots))

buses: ['node']   snapshots: [0]


A generator per technology. `p_nom` is the capacity row's right-hand side; `marginal_cost` is the
objective coefficient; the generator's output `p` will be the decision variable.

In [6]:
n.add("Generator", "solar", bus="node", p_nom=AVAILABLE["solar"], marginal_cost=COST["solar"])
n.add("Generator", "gas",   bus="node", p_nom=AVAILABLE["gas"],   marginal_cost=COST["gas"])
print(n.generators[["bus", "p_nom", "marginal_cost"]])

        bus  p_nom  marginal_cost
name                             
solar  node   40.0           0.00
gas    node  500.0          22.14


The load is the balance row's right-hand side.

In [7]:
n.add("Load", "demand", bus="node", p_set=DEMAND)
print(n.loads[["bus", "p_set"]])

         bus  p_set
name               
demand  node  100.0


Solve. Predict, before running, whether the objective and the bus's marginal price will match Part B
to the cent.

In [8]:
n.optimize(**SOLVE)
pypsa_cost = float(n.objective)
pypsa_price = float(n.buses_t.marginal_price.iloc[0, 0])
print(f"objective ${pypsa_cost:,.2f}")
print(n.generators_t.p.T.rename(columns={0: "MW"}))
print(f"marginal price at the bus: ${pypsa_price:.2f}/MWh")

objective $1,328.40
snapshot    MW
name          
solar     40.0
gas       60.0
marginal price at the bus: $22.14/MWh


## Part D — what PyPSA actually wrote

Ask it. `create_model()` builds the linear program without solving it, and the variables and
constraints can be listed. Match each one to a line of Part B.

In [9]:
lp = n.optimize.create_model()
print("variables PyPSA created:")
for name in lp.variables:
    print(f"  {name:28} shape {lp.variables[name].shape}")
print("\nconstraints PyPSA created:")
for name in lp.constraints:
    print(f"  {name:28} shape {lp.constraints[name].shape}")

variables PyPSA created:
  Generator-p                  shape (1, 2)

constraints PyPSA created:
  Generator-fix-p-lower        shape (1, 2)
  Generator-fix-p-upper        shape (1, 2)
  Bus-nodal_balance            shape (1, 1)


| you wrote | PyPSA wrote |
|---|---|
| `p = m.addVars(...)` | `Generator-p` |
| `p.sum() == DEMAND` | `Bus-nodal_balance` |
| `p[g] <= AVAILABLE[g]` | `Generator-fix-p-upper` |
| `lb=0.0` | `Generator-fix-p-lower` |

Same variables, same rows, same answer. Every component added below writes rows like these.

---

## Part E — now make it too big to check by hand

Twenty-four hours instead of one, demand that moves, solar that follows the sun, two thermal units,
and a battery. The battery is what changes the character of the problem: every other component
decides each hour on its own, but charging at noon is only worth anything given what happens at
seven in the evening. That link between hours is the **state of charge**, and it is why this one
cannot be done in your head.

The day is a table. It was generated from a formula — a 50 MW base, a morning bump at 8, the real
peak at 19, and solar as a half-sine from 6 to 19 — by the package, and the file is what both sides
read. So are the three technologies.

In [10]:
from orteach.energy import load_day_profiles, load_generators

day = load_day_profiles()
techs = load_generators()

print(pd.DataFrame(day).set_index("hour").T.round(2).to_string())
print()
print(f"{'technology':10} {'p_nom MW':>9} {'$/MWh':>7} {'capex $/MW/yr':>14} {'build cap MW':>13}  profile")
for t in techs:
    print(f"{t.name:10} {t.p_nom:9.0f} {t.marginal_cost:7.2f} {t.capital_cost:14,.0f} {t.p_nom_max:13.0f}  {'solar' if t.varies else 'flat'}")
print("\ndemand_mw[19] =", day["demand_mw"][19], "  solar_pu[12] =", day["solar_pu"][12])

# to try a different fleet, edit the loaded table; the change reaches the package check at the bottom:
# techs[1].p_nom = 80.0

hour         0      1      2      3      4      5      6      7      8      9      10     11     12     13     14     15     16     17      18     19     20     21     22     23
demand_mw  50.0  50.01  50.04  50.28  51.25  54.02  59.24  65.24  68.00  65.25  59.31  54.31  52.26  53.27  57.52  65.82  78.34  92.99  105.20  110.0  105.2  92.99  78.34  65.82
solar_pu    0.0   0.00   0.00   0.00   0.00   0.00   0.00   0.24   0.46   0.66   0.82   0.94   0.99   0.99   0.94   0.82   0.66   0.46    0.24    0.0    0.0   0.00   0.00   0.00

technology  p_nom MW   $/MWh  capex $/MW/yr  build cap MW  profile
solar            140    0.00         90,000           400  solar
gas               66   22.14         70,000             0  flat
peaker           100   80.00         25,000             0  flat

demand_mw[19] = 110.0   solar_pu[12] = 0.9927


Look at the two profiles together before building anything. Solar peaks at midday; demand peaks at
19, when solar is gone. There is 140 MW of solar against a 110 MW peak.

Do the arithmetic before you build anything: gas is 66 MW and the battery 40 MW, against an evening
peak of 110 MW. Can the peaker stay off? And what has to happen earlier in the day for that answer
to hold?

The battery's four numbers are knobs. `max_hours=4` means its energy store is four times its power;
the one-way efficiency squared is the round trip; the standing loss is the fraction that leaks away
each hour just sitting there — a small number that does real work below.

In [11]:
BATTERY_MW    = 40.0
BATTERY_HOURS = 4.0
EFFICIENCY    = 0.927        # one way; round trip 0.927^2 = 0.859
STANDING_LOSS = 0.01         # per hour
CYCLE_COST    = 0.01         # $/MWh, a token cost per MWh cycled

T = {t.name: t for t in techs}      # the loaded table, by name

n2 = pypsa.Network()
n2.set_snapshots(day["hour"])
n2.add("Bus", "node")
n2.add("Load", "demand", bus="node", p_set=np.array(day["demand_mw"]))
# three generators with three different stories, so they are written out rather than looped: solar
# follows the sun, gas is the cheap thermal unit, the peaker is the expensive one
n2.add("Generator", "solar", bus="node", p_nom=T["solar"].p_nom,
       marginal_cost=T["solar"].marginal_cost, p_max_pu=np.array(day["solar_pu"]))
n2.add("Generator", "gas", bus="node", p_nom=T["gas"].p_nom, marginal_cost=T["gas"].marginal_cost)
n2.add("Generator", "peaker", bus="node", p_nom=T["peaker"].p_nom, marginal_cost=T["peaker"].marginal_cost)
n2.add("StorageUnit", "battery", bus="node", p_nom=BATTERY_MW, max_hours=BATTERY_HOURS,
       efficiency_store=EFFICIENCY, efficiency_dispatch=EFFICIENCY, standing_loss=STANDING_LOSS,
       cyclic_state_of_charge=True, marginal_cost=CYCLE_COST)
print(n2.generators[["p_nom", "marginal_cost"]])
print(n2.storage_units[["p_nom", "max_hours", "efficiency_store", "standing_loss"]])

        p_nom  marginal_cost
name                        
solar   140.0           0.00
gas      66.0          22.14
peaker  100.0          80.00
         p_nom  max_hours  efficiency_store  standing_loss
name                                                      
battery   40.0        4.0             0.927           0.01


What did PyPSA write this time? Look for the row that ties hour $t$ to hour $t-1$.

In [12]:
lp2 = n2.optimize.create_model()
for name in lp2.constraints:
    print(f"  {name:36} shape {lp2.constraints[name].shape}")

  Generator-fix-p-lower                shape (24, 3)
  Generator-fix-p-upper                shape (24, 3)
  StorageUnit-fix-p_dispatch-lower     shape (24, 1)
  StorageUnit-fix-p_dispatch-upper     shape (24, 1)
  StorageUnit-fix-p_store-lower        shape (24, 1)
  StorageUnit-fix-p_store-upper        shape (24, 1)
  StorageUnit-fix-state_of_charge-lower shape (24, 1)
  StorageUnit-fix-state_of_charge-upper shape (24, 1)
  Bus-nodal_balance                    shape (24, 1)
  StorageUnit-energy_balance           shape (24, 1)


`StorageUnit-energy_balance` is the one you could not have solved on paper — twenty-four rows, each
$E(t) = (1-\ell)\,E(t-1) + \eta_c P_{charge}(t) - P_{discharge}(t)/\eta_d$. The standing loss $\ell$
*multiplies* the energy carried over, so it charges rent on energy for sitting still rather than a
toll for moving.

Predict before running: in which hours will the battery charge, in which will it discharge, and in
how many hours will the expensive unit run at all?

In [13]:
n2.optimize(**SOLVE)
day_cost = float(n2.objective)
prices_hand = [float(v) for v in n2.buses_t.marginal_price["node"]]      # unrounded, for the check below
print(f"day cost: ${day_cost:,.2f}\n")
table = pd.DataFrame({"demand": n2.loads_t.p["demand"], "solar": n2.generators_t.p["solar"],
                      "gas": n2.generators_t.p["gas"], "peaker": n2.generators_t.p["peaker"],
                      "battery": n2.storage_units_t.p["battery"],
                      "state of charge": n2.storage_units_t.state_of_charge["battery"]}).round(1)
table["price"] = n2.buses_t.marginal_price["node"].round(2)
print(table.to_string())

day cost: $18,132.63

          demand  solar   gas  peaker  battery  state of charge  price
snapshot                                                              
0           50.0    0.0  50.0     0.0      0.0              0.0  22.14
1           50.0    0.0  50.0     0.0      0.0              0.0  22.14
2           50.0    0.0  50.0     0.0      0.0              0.0  22.14
3           50.3    0.0  50.3     0.0      0.0              0.0  22.14
4           51.3    0.0  51.3     0.0      0.0              0.0  22.14
5           54.0    0.0  54.0     0.0      0.0              0.0  22.14
6           59.2    0.0  59.2     0.0      0.0              0.0  22.14
7           65.2   33.5  31.7     0.0      0.0              0.0  22.14
8           68.0   65.1   2.9     0.0      0.0              0.0  22.14
9           65.3   65.3   0.0     0.0      0.0              0.0   0.00
10          59.3   59.3   0.0     0.0      0.0              0.0   0.00
11          54.3   56.5   0.0     0.0     -2.2         

Read the table before moving on. The battery column is negative when it charges. Find the hours it
charges in and what the price is then; the hours it discharges into; and the one hour the peaker
runs. Then look at the price in hour 18, and in hours 20 to 22: each is above gas's $22.14, and the
only thing in this system that costs more than gas to run is the peaker, which is off in all four.
What is setting the price there?

In [14]:
charging = [int(h) for h in day["hour"] if table.battery[h] < -tolerance.FEASIBILITY_ATOL]
discharging = [int(h) for h in day["hour"] if table.battery[h] > tolerance.FEASIBILITY_ATOL]
peaker_on = [int(h) for h in day["hour"] if table.peaker[h] > tolerance.FEASIBILITY_ATOL]
print("charging hours   :", charging)
print("discharging hours:", discharging)
print("peaker runs in   :", peaker_on)

charging hours   : [11, 12, 13, 14, 15, 16]
discharging hours: [17, 18, 19, 20, 21, 22]
peaker runs in   : [19]


## Which hours it charges in is not the model's answer

Solar is curtailed in every hour from 9 to 16 — more is available than the system can absorb — so a
MWh put into the battery in any of those hours costs nothing, and swapping one charging hour for
another leaves the total unchanged. Where an exchange is free, the model has not decided anything,
and what comes back is the *algorithm's* answer rather than the problem's.

That is testable. Simplex walks corners and barrier walks the interior, so ask for barrier and
compare. Predict first: which of the numbers printed above do you expect to move?

In [15]:
BARRIER = dict(SOLVE, solver_options=dict(SOLVE["solver_options"], Method=2))
n2.optimize(**BARRIER)
bat2 = n2.storage_units_t.p["battery"]
print(f"barrier:  day cost ${float(n2.objective):,.2f}")
print("  charging   ", [int(h) for h in day["hour"] if bat2[h] < -tolerance.FEASIBILITY_ATOL])
print("  discharging", [int(h) for h in day["hour"] if bat2[h] > tolerance.FEASIBILITY_ATOL])
print(f"  energy returned {float(bat2[bat2 > 0].sum()):.3f} MWh    "
      f"price at 19 ${float(n2.buses_t.marginal_price['node'][19]):.2f}/MWh")
n2.optimize(**SOLVE)                     # back to the default the rest of the notebook uses

barrier:  day cost $18,132.63
  charging    [9, 10, 11, 13, 14, 16]
  discharging [17, 18, 19, 20, 21, 22]
  energy returned 143.024 MWh    price at 19 $80.00/MWh


('ok', 'optimal')

The same cost to the cent, from a different set of charging hours. So those hours are not a property
of this system. What every optimum does agree on is what the model actually determined: the cost, how
much energy the battery gives back and when, the hour the peaker runs, and all twenty-four prices.
Those are what the agreement check at the bottom compares — and the charging hours are checked only
for the property they must have, which is that the model never pays for the energy it stores.

It is tempting to reach for the standing loss as the thing that ought to settle it: energy that leaks
should punish charging early. Predict what happens to the cost and to the schedule, then set it to
zero and re-solve.

In [16]:
n2.storage_units.loc["battery", "standing_loss"] = 0.0
n2.optimize(**SOLVE)
bat0 = n2.storage_units_t.p["battery"]
print(f"no standing loss: day cost ${float(n2.objective):,.2f}   "
      f"energy returned {float(bat0[bat0 > 0].sum()):.3f} MWh")
print("  charging   ", [int(h) for h in day["hour"] if bat0[h] < -tolerance.FEASIBILITY_ATOL])
print("  discharging", [int(h) for h in day["hour"] if bat0[h] > tolerance.FEASIBILITY_ATOL])
n2.storage_units.loc["battery", "standing_loss"] = STANDING_LOSS
n2.optimize(**SOLVE)
print(f"restored:         day cost ${float(n2.objective):,.2f}")

no standing loss: day cost $18,015.42   energy returned 148.320 MWh
  charging    [9, 11, 12, 13, 14]
  discharging [5, 18, 19, 20, 21, 22]


restored:         day cost $18,132.63


The cost moved, so the standing loss is not a tie-breaker at all — it is a real cost, and a sizeable
one. The tie survived it: with the leak gone, energy sits in the battery for free, and now even the
*discharging* hours move between algorithms.

Breaking a tie takes a cost that depends on when energy is held rather than on whether it moves.
Which cost in a real battery behaves that way, and what would adding it do to the schedule above?

---

## Part F — from running a fleet to choosing one

Everything so far took the fleet as given. Capacity expansion asks what to *build*. In PyPSA the
change is one argument: `p_nom_extendable=True`, with a `capital_cost` for each MW built. The
table's capital costs are per MW per year; the snapshots are one representative day, so each is
divided by the days in a year to put a day of capital beside a day of fuel. Get that division wrong
and the model builds almost nothing and burns fuel forever. It is a knob, so it is named here and
passed to the package below rather than typed twice.

The build ceiling is the table's business: solar has one, the two thermal units carry a zero, which
this model reads as "no ceiling".

In [17]:
DAYS_PER_YEAR = 365.0

n3 = pypsa.Network()
n3.set_snapshots(day["hour"])
n3.add("Bus", "node")
n3.add("Load", "demand", bus="node", p_set=np.array(day["demand_mw"]))
for t in techs:
    extra = {}
    if t.varies:                       # only solar follows a profile
        extra["p_max_pu"] = np.array(day["solar_pu"])
    if t.p_nom_max > 0:                # a zero in the table's build-ceiling column means "no ceiling"
        extra["p_nom_max"] = t.p_nom_max
    n3.add("Generator", t.name, bus="node", p_nom_extendable=True, marginal_cost=t.marginal_cost,
           capital_cost=t.capital_cost / DAYS_PER_YEAR, **extra)
n3.add("StorageUnit", "battery", bus="node", p_nom=BATTERY_MW, max_hours=BATTERY_HOURS,
       efficiency_store=EFFICIENCY, efficiency_dispatch=EFFICIENCY, standing_loss=STANDING_LOSS,
       cyclic_state_of_charge=True, marginal_cost=CYCLE_COST)

lp3 = n3.optimize.create_model()
print("new variables:", [v for v in lp3.variables if "p_nom" in v])

new variables: ['Generator-p_nom']


`Generator-p_nom` is now a variable. In Part E it was a number in a table. Predict before running:
solar is free to run — will the model build a lot of it?

In [18]:
n3.optimize(**SOLVE)
built = {g: float(v) for g, v in n3.generators.p_nom_opt.items()}
print(f"day cost ${float(n3.objective):,.2f}\n")
for g, mw in built.items():
    print(f"  {g:8} {mw:7.1f} MW built")

day cost $51,337.33

  solar        0.0 MW built
  gas         74.4 MW built
  peaker       0.0 MW built


## The envelope calculation, and where it goes wrong

Work out by hand why the model would not touch free fuel: one MW of solar makes so many MWh a day,
each displacing gas at $22.14, and that is worth so much a year. Compare it with the capital cost the
table charges.

In [19]:
SOLAR_CAPEX = next(t.capital_cost for t in techs if t.varies)
GAS_COST = next(t.marginal_cost for t in techs if t.name == "gas")

energy_per_mw = float(np.sum(day["solar_pu"]))            # MWh per MW per day
value_per_day = energy_per_mw * GAS_COST
breakeven_hand = value_per_day * DAYS_PER_YEAR
print(f"capacity factor              {energy_per_mw / 24:.3f}")
print(f"energy from 1 MW of solar    {energy_per_mw:.2f} MWh/day")
print(f"value if it displaces gas    ${value_per_day:.2f}/day")
print(f"worth building only below    ${breakeven_hand:,.0f}/MW/yr")
print(f"the table charges            ${SOLAR_CAPEX:,.0f}/MW/yr")

capacity factor              0.343
energy from 1 MW of solar    8.24 MWh/day
value if it displaces gas    $182.34/day
worth building only below    $66,553/MW/yr
the table charges            $90,000/MW/yr


The envelope says no solar at the table's price, and the model agreed. Now find the price at which
the model *does* build it, by asking it — the same expansion network at a sweep of solar capital
costs. Predict first: will the model's break-even be above or below the envelope's?

In [20]:
CAPEX_SWEEP = (90_000, 83_000, 75_000, 66_000, 60_000)
solar_hand = {}
print(f"{'solar capex':>14}  {'solar built':>12}")
for capex in CAPEX_SWEEP:
    k = pypsa.Network()
    k.set_snapshots(day["hour"])
    k.add("Bus", "node")
    k.add("Load", "demand", bus="node", p_set=np.array(day["demand_mw"]))
    for t in techs:
        extra = {}
        if t.varies:
            extra["p_max_pu"] = np.array(day["solar_pu"])
        if t.p_nom_max > 0:
            extra["p_nom_max"] = t.p_nom_max
        k.add("Generator", t.name, bus="node", p_nom_extendable=True, marginal_cost=t.marginal_cost,
              capital_cost=(capex if t.varies else t.capital_cost) / DAYS_PER_YEAR, **extra)
    k.add("StorageUnit", "battery", bus="node", p_nom=BATTERY_MW, max_hours=BATTERY_HOURS,
          efficiency_store=EFFICIENCY, efficiency_dispatch=EFFICIENCY, standing_loss=STANDING_LOSS,
          cyclic_state_of_charge=True, marginal_cost=CYCLE_COST)
    k.optimize(**SOLVE)
    solar_hand[capex] = float(k.generators.p_nom_opt["solar"])
    print(f"  ${capex:>10,}/yr  {solar_hand[capex]:>9.1f} MW")

   solar capex   solar built


  $    90,000/yr        0.0 MW


  $    83,000/yr        3.5 MW


  $    75,000/yr       29.3 MW


  $    66,000/yr       78.2 MW


  $    60,000/yr       92.9 MW


The model builds solar well above the price the envelope allowed. The envelope valued solar at the
fuel it displaces and nothing else. What else does an MW of solar arriving in the afternoon, ahead of
the evening ramp, do to the rest of the plan — and why can no back-of-envelope calculation see it?

---

# Now the streamlined version

Four networks built by hand from the same tables, so the package owns the constructions:
`energy.dispatch_lp` and `energy.one_hour_network` for the hour, `energy.day_network` with
`expand=` for the day and the build decision, `energy.solve_day` to read a solve back, and
`energy.solar_built` for the sweep. `energy.envelope_breakeven` is the hand calculation.

In [21]:
from orteach import energy
from orteach.tolerance import AGREEMENT_RTOL, rel_diff

battery = energy.Battery(BATTERY_MW, BATTERY_HOURS, EFFICIENCY, STANDING_LOSS, CYCLE_COST)
techs_hour = {g: (AVAILABLE[g], COST[g]) for g in AVAILABLE}

pkg_lp = energy.dispatch_lp(techs_hour, DEMAND, env=env)
pkg_hour = energy.one_hour_network(techs_hour, DEMAND)
pkg_hour.optimize(**SOLVE)
pkg_day = energy.solve_day(energy.day_network(day, techs, battery), env)
pkg_build = energy.solve_day(energy.day_network(day, techs, battery, expand=True, days_per_year=DAYS_PER_YEAR), env)
pkg_sweep = {capex: energy.solar_built(day, techs, battery, capex, env, days_per_year=DAYS_PER_YEAR)
             for capex in CAPEX_SWEEP}

print(f"one hour: LP ${pkg_lp.objective:,.2f} at ${pkg_lp.price:.2f}/MWh   PyPSA ${float(pkg_hour.objective):,.2f}")
print(f"day: ${pkg_day.objective:,.2f}   charging {pkg_day.charging_hours}   peaker {pkg_day.peaker_hours}")
built_pkg = {k: round(v, 1) for k, v in pkg_build.built.items()}
print(f"build at the table's prices: {built_pkg}")
print("solar built by capex:", {c: round(v, 1) for c, v in pkg_sweep.items()})
print(f"envelope break-even ${energy.envelope_breakeven(day, GAS_COST, DAYS_PER_YEAR):,.0f}/MW/yr")

one hour: LP $1,328.40 at $22.14/MWh   PyPSA $1,328.40
day: $18,132.63   charging [11, 12, 13, 14, 15, 16]   peaker [19]
build at the table's prices: {'solar': 0.0, 'gas': 74.4, 'peaker': 0.0}
solar built by capex: {90000: 0.0, 83000: 3.5, 75000: 29.3, 66000: 78.2, 60000: 92.9}
envelope break-even $66,553/MW/yr


## The agreement assertion

Four ways of getting the hour's cost and price, the day's cost and prices, the build decision and the
whole sweep — hand-built against the package. Every solve on both sides went through Gurobi at the
package's tightened tolerances, so agreement to `AGREEMENT_RTOL` is a claim the computation supports.

The schedule is compared for what every optimum shares and no more. Discharging hours and the peaker
hour are the model's answer, so they must match exactly. The charging hours are not, so the check
asks only that the package never pays for the energy it stores — the property that makes the whole
set of optima equivalent.

In [22]:
checks = [("hour cost, gurobipy", lp_cost, pkg_lp.objective),
          ("hour price, gurobipy", lp_price, pkg_lp.price),
          ("hour cost, PyPSA", pypsa_cost, float(pkg_hour.objective)),
          ("hour price, PyPSA", pypsa_price, float(pkg_hour.buses_t.marginal_price.iloc[0, 0])),
          ("hour: gurobipy vs PyPSA", lp_cost, pypsa_cost),
          ("day cost", day_cost, pkg_day.objective),
          ("envelope break-even", breakeven_hand, energy.envelope_breakeven(day, GAS_COST, DAYS_PER_YEAR))]
for g in built:
    checks.append((f"built {g}", built[g], pkg_build.built[g]))
for capex in CAPEX_SWEEP:
    checks.append((f"solar at {capex:,}", solar_hand[capex], pkg_sweep[capex]))
for h in day["hour"]:
    checks.append((f"price at hour {h}", prices_hand[h], pkg_day.prices[h]))
assert discharging == pkg_day.discharging_hours, "discharging hours differ"
assert peaker_on == pkg_day.peaker_hours, "peaker hours differ"
assert all(pkg_day.prices[h] < tolerance.FEASIBILITY_ATOL for h in pkg_day.charging_hours), \
    "the package charged the battery in an hour that was not free"

worst = max(rel_diff(h, k) for _, h, k in checks)
print(f"{len(checks)} comparisons, plus the schedule invariants")
for name, hand, packaged in checks[:5]:
    print(f"  {name:24} hand {hand:12.4f}   package {packaged:12.4f}   rel {rel_diff(hand, packaged):.2e}")
print("  ...")
assert worst < AGREEMENT_RTOL, f"notebook and package disagree by {worst:.2e}"
print(f"\nnotebook and package agree to {worst:.1e}")

39 comparisons, plus the schedule invariants
  hour cost, gurobipy      hand    1328.4000   package    1328.4000   rel 0.00e+00
  hour price, gurobipy     hand      22.1400   package      22.1400   rel 0.00e+00
  hour cost, PyPSA         hand    1328.4000   package    1328.4000   rel 0.00e+00
  hour price, PyPSA        hand      22.1400   package      22.1400   rel 0.00e+00
  hour: gurobipy vs PyPSA  hand    1328.4000   package    1328.4000   rel 0.00e+00
  ...

notebook and package agree to 0.0e+00


---

## Where to take this next

- Add a coal plant — 200 MW available at $19.90/MWh — to *both* one-hour models, and confirm they
  still agree. Predict, before running, whether the marginal price goes up or down, and why.
- Set the battery's power to zero in the day model and re-solve. How much of the day's cost was the
  battery saving, and in which hours?
- Repeat the capital-cost sweep for the peaker instead of solar. Does the model value it above or
  below its energy displacement, and why is the answer the opposite sign to solar's?
- Cap solar's build at 50 MW and re-run the sweep at $60,000/yr. The model builds all 50. Which is
  binding — the cap or the price — and how would you tell?